In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('D:\Dev\Point-Cloud-Registration\source')

import pickle
import os
from copy import deepcopy
import pandas as pd
import numpy as np
from glob import glob
from pathlib import Path
import matplotlib.pyplot as plt

from cylinder_alignment.bundle_station import BundleStation
from cylinder_extraction.station import Station
from cylinder_extraction.station import Station
from cylinder_alignment.station_alignment import StationAlignment

import plotly.express as px

In [2]:
data_dir = Path('D:/REGISTRATION DATA/External/JB-453 60ft EXT')
filename_template = '{}.e57'
ptc_col_names =  ["x","y","z"]

# Read extracted cylinder data

In [3]:
source_files = glob(str(data_dir / '*.e57'))
station_names = []
for fp in source_files:
    if 'roof' in fp.lower():
        continue
    fn = Path(fp).stem
    # station id
    sn = fn[-4:]
    print(sn)
    station_names.append(sn)
stations = []
for sn in station_names:
    pkl_fp = os.path.join(
        data_dir,
        f'{sn}-output',
        f'{sn}.pkl'
    )
    if os.path.exists(pkl_fp):
        with open(pkl_fp, 'rb') as f:
            st = pickle.load(f)
            stations.append(st)


9001
9002
9003
9004
9005
9006
9007
9008
9009
9010
9011


In [5]:
summary = []
for s in stations:
    # print('======')
    # print(s)
    s.update_filepath(data_dir, num_keep_level=1)
    for c in s.refined_cylinders:
        # update path
        c.update_filepath(data_dir, num_keep_level=2)
        # print(c)
        summary.append({
            'station': s.name,
            'cylinder': c.id,
            'radius': c.radius
        })
summary_df = pd.DataFrame(summary)
summary_df.head()

,station,cylinder,radius
0,9001,2,21.835470
1,9001,3,9.114079
2,9001,5,11.919918
3,9002,3,6.381104
4,9002,5,9.112638


In [6]:
px.scatter(
    summary_df[summary_df.radius < 15], y='radius', x='station', 
    color='station', hover_data=summary_df.columns
)

# Pairwise alignment

In [7]:
num_station = len(stations)
alignment = []
for i in range(num_station):
    st0 = stations[i]
    # if st0.name != '089':
    #     continue
    for j in range(i+1, num_station):
        st1 = stations[j]
        # if st1.name != '093':
        #     continue
        align = StationAlignment(st0, st1)
        align.align(max_delta_radius=1, top_surface=False)
        alignment.append(align)

Align station 9001 and 9002:
3
21.83547019958496 23.749773025512695 1.9143028259277344
Align station 9001 and 9003:
3
21.83547019958496 7.3231658935546875 14.512304306030273
Align station 9001 and 9004:
3
[[ 0.90755855 -0.41992556  0.        ]
 [ 0.41992556  0.90755855  0.        ]
 [ 0.          0.          1.        ]]
[[-2.06014813]
 [ 6.21130941]
 [ 0.        ]]
0.10584963788901404
3 7
2 9
5 10
[[  3.           9.11407852 -12.08520603   8.53748035   0.        ]
 [  2.          21.8354702   -5.26596546  67.18228912   0.        ]
 [  5.          11.91991806  24.01077652   9.06159973   0.        ]]
[[ 7.          9.07696724 -8.05715752  6.54685688  0.        ]
 [ 9.         21.51376152 22.56920624 56.54598618  0.        ]
 [10.         11.73355579 24.91804886 -8.45203209  0.        ]]
Align station 9001 and 9005:
3
9.114078521728516 7.317351818084717 1.7967267036437988
Align station 9001 and 9006:
3
9.114078521728516 7.336071014404297 1.7780075073242188
Align station 9001 and 9007:
3


In [8]:
ori_stations = deepcopy(stations)

In [9]:
stations = deepcopy(ori_stations)

# Align all stations

In [10]:
# all stations
stations = deepcopy(ori_stations)
bundle = BundleStation(stations)
# 
bundle.rough_alignment()
# print(bundle.rough_align)
# target cylinder
bundle.guess_target_cylinder()

In [11]:
bundle.cluster_cylinder()

In [12]:
with open(os.path.join(data_dir, 'tank_info.pkl'), 'wb') as f:
    pickle.dump(bundle.tanks_info, f)

# Cylinder vertical axes alignment

In [13]:
for s in stations:
    s.read_points()
    for c in s.refined_cylinders:
        c.read_points()
        c.extend_ground()
        c.make_vertical()
        c.translate_base()
        # c.translate_origin()
        # c.write_points()

D:\REGISTRATION DATA\External\JB-453 60ft EXT Jb453 Scan_TMG_TK_9001.e57
[0 0 0 ... 0 0 0]
(22851338, 3)
(375771, 3)
(22851338, 3)
(5400935, 3)
(22851338, 3)
(1562545, 3)
D:\REGISTRATION DATA\External\JB-453 60ft EXT Jb453 Scan_TMG_TK_9002.e57
[0 0 0 ... 0 0 0]
(21479841, 3)
(17541, 3)
(21479841, 3)
(5245767, 3)
(21479841, 3)
(803432, 3)
(21479841, 3)
(40706, 3)
D:\REGISTRATION DATA\External\JB-453 60ft EXT Jb453 Scan_TMG_TK_9003.e57
[0 0 0 ... 0 0 0]
(28325287, 3)
(226729, 3)
(28325287, 3)
(8139007, 3)
(28325287, 3)
(417165, 3)
D:\REGISTRATION DATA\External\JB-453 60ft EXT Jb453 Scan_TMG_TK_9004.e57
[0 0 0 ... 0 0 0]
(31065774, 3)
(34862, 3)
(31065774, 3)
(475473, 3)
(31065774, 3)
(26021, 3)
(31065774, 3)
(5203867, 3)
(31065774, 3)
(79638, 3)
(31065774, 3)
(52456, 3)
D:\REGISTRATION DATA\External\JB-453 60ft EXT Jb453 Scan_TMG_TK_9005.e57
[0 0 0 ... 0 0 0]
(24887914, 3)
(899111, 3)
(24887914, 3)
(5608250, 3)
(24887914, 3)
(26754, 3)
(24887914, 3)
(483905, 3)
D:\REGISTRATION DATA\Exter

In [14]:
# with open(os.path.join(data_dir, 'tank_centers_aligned.pkl'), 'wb') as f:
#     pickle.dump(bundle.tanks_info['centers_xy'], f)

In [15]:
bundle.write_all_tanks(data_dir, surrounding=False)

Writing tank 0
Writing tank 1
Writing tank 2
Writing tank 3
Writing tank 4
Writing tank 5
Writing tank 6
Writing tank 7


In [16]:
center_df = bundle.tanks_info['centers_xy']
# center_df.head()

In [17]:
px.scatter(
    center_df[center_df.label != '-1'], 
    x='x', y='y', 
    color='label', hover_data=['station', 'cylinder id', 'radius', 'x', 'y'], 
    width=700, height=700)